# 06 — CERT-FR CTI Report Scraping

**C1 Source Type:** `Scraping`

---

## Objective

Scrape phishing indicators from [CERT-FR](https://www.cert.ssi.gouv.fr/) — France's official
Computer Emergency Response Team (part of ANSSI). CERT-FR publishes **Cyber Threat Intelligence
(CTI) reports** and **Indicators of Compromise (IOCs)** for campaigns targeting French entities.

### Why CERT-FR?

| Criterion | Detail |
|-----------|--------|
| Format | HTML pages + PDF reports |
| Coverage | French-targeted campaigns (APT28, Nobelium, credential harvesting) |
| French signal | 100% — all reports are about attacks on French entities |
| C1 requirement | Satisfies *"scraping"* extraction type |
| Cost | Free, public |

### Pipeline

```
cert.ssi.gouv.fr/cti/ → Scrape report list → Extract IOCs per report
                       → Download linked PDFs → Extract text with pdfplumber
                       → Structure as JSON/CSV
```

### Output

- `data/raw/scraping/certfr/certfr_cti_reports_<N>_<date>.csv`
- `data/raw/scraping/certfr/certfr_iocs_<N>_<date>.csv`

In [1]:
# ── Imports & Constants ──────────────────────────────────────────────
from __future__ import annotations

import re
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urljoin

import httpx
import pandas as pd
from bs4 import BeautifulSoup

# ── Configuration ────────────────────────────────────────────────────
CERTFR_BASE_URL: str = "https://www.cert.ssi.gouv.fr"
CTI_INDEX_URL: str = f"{CERTFR_BASE_URL}/cti/"
IOC_INDEX_URL: str = f"{CERTFR_BASE_URL}/ioc/"

OUTPUT_DIR: Path = Path("data/raw/scraping/certfr")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Polite scraping: 1-2s between requests
REQUEST_DELAY: float = 1.5
REQUEST_TIMEOUT: int = 30

# Regex patterns for IOC extraction
EMAIL_PATTERN: re.Pattern = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")
DOMAIN_PATTERN: re.Pattern = re.compile(
    r"\b(?:[a-zA-Z0-9](?:[a-zA-Z0-9-]{0,61}[a-zA-Z0-9])?\.)+"
    r"(?:fr|com|net|org|info|eu|io|xyz|top|ru|cn)\b"
)
IP_PATTERN: re.Pattern = re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b")
HASH_PATTERN: re.Pattern = re.compile(r"\b[a-fA-F0-9]{32,64}\b")
SUBJECT_PATTERN: re.Pattern = re.compile(
    r"(?:objet|sujet|subject)\s*[:：]\s*(.+)", re.IGNORECASE
)

HEADERS: dict[str, str] = {
    "User-Agent": "Mozilla/5.0 (sicurre-research; academic)",
    "Accept-Language": "fr-FR,fr;q=0.9",
}

print(f"CTI index   : {CTI_INDEX_URL}")
print(f"IOC index   : {IOC_INDEX_URL}")
print(f"Output dir  : {OUTPUT_DIR.resolve()}")
print(f"Delay       : {REQUEST_DELAY}s between requests")

CTI index   : https://www.cert.ssi.gouv.fr/cti/
IOC index   : https://www.cert.ssi.gouv.fr/ioc/
Output dir  : /Users/michaeladebayo/Documents/Simplon/brief_projects/sicurre/data/raw/scraping/certfr
Delay       : 1.5s between requests


## 1. Scrape CTI Report List

The CTI index page lists all published threat intelligence reports. Each report has:
- A unique CERTFR identifier (e.g., `CERTFR-2025-CTI-007`)
- A title (in French)
- A publication date
- A detail page URL

We paginate through the index to collect all report metadata.

In [2]:
def scrape_certfr_index(index_url: str, *, max_pages: int = 20) -> list[dict]:
    """Scrape paginated CERT-FR index (CTI or IOC) and return report metadata."""
    reports: list[dict] = []
    page: int = 1
    
    with httpx.Client(timeout=REQUEST_TIMEOUT, follow_redirects=True, headers=HEADERS) as client:
        while page <= max_pages:
            url: str = f"{index_url}" if page == 1 else f"{index_url}page/{page}/"
            print(f"  Page {page}: {url}", end=" ")
            
            try:
                resp = client.get(url)
                if resp.status_code == 404:
                    print("→ 404 (end of pages)")
                    break
                resp.raise_for_status()
            except httpx.HTTPError as e:
                print(f"→ Error: {e}")
                break
            
            soup = BeautifulSoup(resp.text, "html.parser")
            
            # CERT-FR uses article tags or specific CSS classes
            articles = soup.select("article.cert-alert, article.item")
            if not articles:
                # Fallback: look for links matching CERTFR pattern
                articles = soup.find_all("a", href=re.compile(r"/cti/CERTFR|/ioc/CERTFR"))
            
            if not articles:
                print("→ 0 articles (end)")
                break
            
            count: int = 0
            for article in articles:
                # Extract link
                link_tag = article if article.name == "a" else article.find("a")
                if not link_tag or not link_tag.get("href"):
                    continue
                
                href: str = urljoin(CERTFR_BASE_URL, link_tag["href"])
                title: str = link_tag.get_text(strip=True)
                
                # Extract CERTFR ID from URL
                certfr_match = re.search(r"CERTFR-\d{4}-(?:CTI|IOC)-\d+", href)
                certfr_id: str = certfr_match.group(0) if certfr_match else ""
                
                # Extract date if present
                date_tag = article.find("time") if article.name != "a" else None
                pub_date: str = date_tag.get("datetime", "") if date_tag else ""
                
                reports.append({
                    "certfr_id": certfr_id,
                    "title": title,
                    "url": href,
                    "pub_date": pub_date,
                })
                count += 1
            
            print(f"→ {count} reports")
            page += 1
            time.sleep(REQUEST_DELAY)
    
    return reports


# ── Scrape CTI index ──────────────────────────────────────────────────
print("Scraping CTI report index...")
cti_reports: list[dict] = scrape_certfr_index(CTI_INDEX_URL)
print(f"\nTotal CTI reports found: {len(cti_reports)}")

Scraping CTI report index...
  Page 1: https://www.cert.ssi.gouv.fr/cti/ → 10 reports
  Page 2: https://www.cert.ssi.gouv.fr/cti/page/2/ → 10 reports
  Page 3: https://www.cert.ssi.gouv.fr/cti/page/3/ → 10 reports
  Page 4: https://www.cert.ssi.gouv.fr/cti/page/4/ → 10 reports
  Page 5: https://www.cert.ssi.gouv.fr/cti/page/5/ → 10 reports
  Page 6: https://www.cert.ssi.gouv.fr/cti/page/6/ → 10 reports
  Page 7: https://www.cert.ssi.gouv.fr/cti/page/7/ → 10 reports
  Page 8: https://www.cert.ssi.gouv.fr/cti/page/8/ → 6 reports
  Page 9: https://www.cert.ssi.gouv.fr/cti/page/9/ → 0 articles (end)

Total CTI reports found: 76


In [3]:
# ── Also scrape IOC index ─────────────────────────────────────────────
print("Scraping IOC index...")
ioc_reports: list[dict] = scrape_certfr_index(IOC_INDEX_URL)
print(f"\nTotal IOC reports found: {len(ioc_reports)}")

Scraping IOC index...
  Page 1: https://www.cert.ssi.gouv.fr/ioc/ → 10 reports
  Page 2: https://www.cert.ssi.gouv.fr/ioc/page/2/ → 5 reports
  Page 3: https://www.cert.ssi.gouv.fr/ioc/page/3/ → 0 articles (end)

Total IOC reports found: 15


In [4]:
# ── Preview ───────────────────────────────────────────────────────────
df_cti: pd.DataFrame = pd.DataFrame(cti_reports)
df_ioc: pd.DataFrame = pd.DataFrame(ioc_reports)

print(f"CTI reports: {len(df_cti)}")
if not df_cti.empty:
    display(df_cti.head(5))

print(f"\nIOC reports: {len(df_ioc)}")
if not df_ioc.empty:
    display(df_ioc.head(5))

CTI reports: 76


,certfr_id,title,url,pub_date
0,CERTFR-2026-CTI-001,L’intelligence artificielle générative face au...,https://www.cert.ssi.gouv.fr/cti/CERTFR-2026-C...,
1,CERTFR-2025-CTI-013,🇬🇧 Mobile phones : Threat landscape since 2015,https://www.cert.ssi.gouv.fr/cti/CERTFR-2025-C...,
2,CERTFR-2025-CTI-012,Téléphones mobiles : État de la menace depuis ...,https://www.cert.ssi.gouv.fr/cti/CERTFR-2025-C...,
3,CERTFR-2025-CTI-011,Opération ENDGAME de novembre 2025,https://www.cert.ssi.gouv.fr/cti/CERTFR-2025-C...,
4,CERTFR-2025-CTI-010,Campagne de notifications de menace envoyée pa...,https://www.cert.ssi.gouv.fr/cti/CERTFR-2025-C...,



IOC reports: 15


,certfr_id,title,url,pub_date
0,CERTFR-2024-IOC-002,Codes malveillants utilisés à des fins destruc...,https://www.cert.ssi.gouv.fr/ioc/CERTFR-2024-I...,
1,CERTFR-2024-IOC-001,🇬🇧 Malicious activities linked to the Nobelium...,https://www.cert.ssi.gouv.fr/ioc/CERTFR-2024-I...,
2,CERTFR-2023-IOC-001,FIN 12 : Un groupe cybercriminel aux multiples...,https://www.cert.ssi.gouv.fr/ioc/CERTFR-2023-I...,
3,CERTFR-2022-IOC-001,🇫🇷/🇬🇧 Feed MISP public,https://www.cert.ssi.gouv.fr/ioc/CERTFR-2022-I...,
4,CERTFR-2021-IOC-005,🇫🇷/🇬🇧 Campagnes d'hameçonnage du mode opératoi...,https://www.cert.ssi.gouv.fr/ioc/CERTFR-2021-I...,


## 2. Extract IOCs from Report Detail Pages

For each report, we visit the detail page and extract:
- **Domains** associated with phishing campaigns
- **Email addresses** (sender spoofing patterns)
- **IP addresses** (infrastructure)
- **Subject lines** (French phishing email subjects)
- **Hashes** (malware indicators)
- **Report text** (full French-language content for NLP analysis)

In [5]:
def extract_iocs_from_page(html: str) -> dict:
    """Extract IOCs and text content from a CERT-FR report page."""
    soup = BeautifulSoup(html, "html.parser")
    
    # Get main content area
    content_div = soup.select_one(".content, article, main, .cert-alert-body")
    text: str = content_div.get_text(separator="\n", strip=True) if content_div else soup.get_text(separator="\n", strip=True)
    
    # Extract IOCs via regex
    domains: list[str] = list(set(DOMAIN_PATTERN.findall(text)))
    emails: list[str] = list(set(EMAIL_PATTERN.findall(text)))
    ips: list[str] = list(set(IP_PATTERN.findall(text)))
    hashes: list[str] = list(set(HASH_PATTERN.findall(text)))
    
    # Extract French phishing subject lines
    subjects: list[str] = SUBJECT_PATTERN.findall(text)
    
    # Check if report mentions phishing
    phishing_keywords: list[str] = [
        "hameçonnage", "phishing", "spear-phishing", "credential",
        "courriel", "email malveillant", "ingénierie sociale",
        "usurpation", "chantage", "rançon",
    ]
    is_phishing_related: bool = any(kw in text.lower() for kw in phishing_keywords)
    
    return {
        "text": text[:5000],  # Truncate for storage
        "text_length": len(text),
        "domains": domains,
        "emails": emails,
        "ips": ips,
        "hashes": hashes,
        "subject_lines": subjects,
        "n_domains": len(domains),
        "n_emails": len(emails),
        "n_ips": len(ips),
        "n_hashes": len(hashes),
        "is_phishing_related": is_phishing_related,
    }


print("IOC extraction function defined.")
print("Phishing keywords: hameçonnage, phishing, spear-phishing, credential, ...")

IOC extraction function defined.
Phishing keywords: hameçonnage, phishing, spear-phishing, credential, ...


In [6]:
# ── Fetch detail pages and extract IOCs ───────────────────────────────
# Combine CTI + IOC reports, prioritize CTI (more content)
all_reports: list[dict] = cti_reports + ioc_reports

# Deduplicate by certfr_id
seen_ids: set[str] = set()
unique_reports: list[dict] = []
for r in all_reports:
    rid: str = r.get("certfr_id", r.get("url", ""))
    if rid and rid not in seen_ids:
        seen_ids.add(rid)
        unique_reports.append(r)

print(f"Unique reports to fetch: {len(unique_reports)}")
print(f"Fetching detail pages (with {REQUEST_DELAY}s delay)...\n")

enriched: list[dict] = []
errors: list[str] = []

with httpx.Client(timeout=REQUEST_TIMEOUT, follow_redirects=True, headers=HEADERS) as client:
    for i, report in enumerate(unique_reports):
        url: str = report["url"]
        certfr_id: str = report.get("certfr_id", f"unknown-{i}")
        
        if (i + 1) % 10 == 0 or i == 0:
            print(f"  [{i+1}/{len(unique_reports)}] {certfr_id}")
        
        try:
            resp = client.get(url)
            resp.raise_for_status()
            iocs: dict = extract_iocs_from_page(resp.text)
            enriched.append({**report, **iocs, "source": "certfr_scraping"})
        except httpx.HTTPError as e:
            errors.append(f"{certfr_id}: {e}")
        
        time.sleep(REQUEST_DELAY)

print(f"\nFetched      : {len(enriched)}")
print(f"Errors       : {len(errors)}")
if errors:
    for err in errors[:5]:
        print(f"  - {err}")

Unique reports to fetch: 91
Fetching detail pages (with 1.5s delay)...

  [1/91] CERTFR-2026-CTI-001
  [10/91] CERTFR-2025-CTI-005
  [20/91] CERTFR-2024-CTI-006
  [30/91] CERTFR-2023-CTI-006
  [40/91] CERTFR-2022-CTI-003
  [50/91] CERTFR-2021-CTI-006
  [60/91] CERTFR-2020-CTI-008
  [70/91] CERTFR-2019-CTI-007
  [80/91] CERTFR-2022-IOC-001
  [90/91] CERTFR-2020-IOC-002

Fetched      : 91
Errors       : 0


In [7]:
# ── Build results DataFrame ───────────────────────────────────────────
df_enriched: pd.DataFrame = pd.DataFrame(enriched)

if not df_enriched.empty:
    phishing_mask = df_enriched["is_phishing_related"]
    print(f"Total reports        : {len(df_enriched)}")
    print(f"Phishing-related     : {phishing_mask.sum()} ({phishing_mask.sum() / len(df_enriched) * 100:.1f}%)")
    print(f"Avg IOCs per report  : domains={df_enriched['n_domains'].mean():.1f}, "
          f"emails={df_enriched['n_emails'].mean():.1f}, "
          f"IPs={df_enriched['n_ips'].mean():.1f}")
    print(f"\nSubject lines found  : {df_enriched['subject_lines'].apply(len).sum()}")
else:
    print("No reports enriched.")

Total reports        : 91
Phishing-related     : 37 (40.7%)
Avg IOCs per report  : domains=0.2, emails=0.0, IPs=0.0

Subject lines found  : 91


## 3. Export

In [8]:
# ── Export reports CSV ─────────────────────────────────────────────────
timestamp: str = datetime.now(timezone.utc).strftime("%Y%m%d")

if not df_enriched.empty:
    # Main reports file (flattened — lists as pipe-separated)
    df_export: pd.DataFrame = df_enriched.copy()
    for col in ["domains", "emails", "ips", "hashes", "subject_lines"]:
        df_export[col] = df_export[col].apply(lambda x: "|".join(x) if isinstance(x, list) else x)
    
    report_path: Path = OUTPUT_DIR / f"certfr_cti_reports_{len(df_export)}_{timestamp}.csv"
    df_export.to_csv(report_path, index=False, encoding="utf-8")
    
    size_mb: float = report_path.stat().st_size / (1024 * 1024)
    print(f"Reports CSV  : {report_path}")
    print(f"Rows         : {len(df_export):,}")
    print(f"Size         : {size_mb:.2f} MB")
    
    # Phishing-only subset
    df_phishing: pd.DataFrame = df_export[df_export["is_phishing_related"]]
    if not df_phishing.empty:
        phish_path: Path = OUTPUT_DIR / f"certfr_phishing_{len(df_phishing)}_{timestamp}.csv"
        df_phishing.to_csv(phish_path, index=False, encoding="utf-8")
        print(f"\nPhishing CSV : {phish_path}")
        print(f"Rows         : {len(df_phishing):,}")
else:
    print("No data to export.")

Reports CSV  : data/raw/scraping/certfr/certfr_cti_reports_91_20260301.csv
Rows         : 91
Size         : 0.20 MB

Phishing CSV : data/raw/scraping/certfr/certfr_phishing_37_20260301.csv
Rows         : 37


## 4. Summary

### What this notebook demonstrates (C1)

| Criterion | Evidence |
|-----------|----------|
| **Source type** | Web scraping (HTML parsing) |
| **Target** | CERT-FR official CTI and IOC reports |
| **Extraction** | BeautifulSoup HTML parsing + regex IOC extraction |
| **Polite scraping** | 1.5s delay between requests, User-Agent header |
| **IOC types** | Domains, emails, IPs, hashes, French subject lines |
| **Output** | Structured CSV with full provenance |

### Value for Sicurre

- **Subject lines** → French phishing template patterns for CamemBERTv2 training
- **Domains** → URL reputation features (cross-ref with PhishTank)
- **Report text** → Rich French-language phishing descriptions for NLP
- **Campaign patterns** → Understanding of French-specific attack vectors

### Limitations

- Reports describe campaigns but don't contain full email bodies
- IOC extraction is heuristic-based (regex) — some false positives possible
- PDF reports (linked from detail pages) would yield more content — see follow-up with `pdfplumber`